## Load Users Data from Bronze Layer

This cell loads the raw users data from the bronze layer and fixes data quality issues in the `signup_date` column by:
* Replacing dots with slashes in the date format
* Converting to proper date type using the M/d/yy format

In [0]:
df = spark.table("databrickscourse.google_sheets.users_dirty")

# Display unique values and types in signup_date to identify issues
display(df.select("signup_date").distinct())

# Fix signup_date: first replace dots with slashes, then convert to proper date format
from pyspark.sql.functions import to_date, col, regexp_replace

df_clean = df.withColumn(
    "signup_date",
    to_date(regexp_replace(col("signup_date"), "\\.", "/"), "M/d/yy")
)

display(df_clean)

signup_date
6/10/24
2/7/22
12/11/22
7/11/23
5/23/24
6/2/22
2.29.24
5/5/22
7/15/23
11/2/24


_row,_fivetran_synced,country,signup_date,user_id,referral_source,last_name,first_name,email
22,2026-05-02T10:07:01.276Z,AU,2024-06-10,USR_1021,social_media,Rodriguez,Edward,edward.rodriguez21@example.com
27,2026-05-02T10:07:01.277Z,US,2022-02-07,USR_1026,partner,Ramirez,Emma,emma.ramirez26@example.com
32,2026-05-02T10:07:01.277Z,AU,2022-12-11,USR_1031,partner,Lopez,Barbara,barbara.lopez31@example.com
37,2026-05-02T10:07:01.277Z,AU,2023-07-11,USR_1036,partner,Garcia,Emma,emma.garcia36@example.com
5,2026-05-02T10:07:01.276Z,UK,2024-05-23,USR_1004,social_media,Miller,Emma,emma.miller4@example.com
42,2026-05-02T10:07:01.277Z,US,2022-06-02,USR_1041,referral,Davis,Patricia,patricia.davis41@example.com
10,2026-05-02T10:07:01.276Z,US,2024-02-29,USR_1009,partner,Jones,Edward,edward.jones9@example.com
47,2026-05-02T10:07:01.277Z,US,2022-05-05,USR_1046,organic,Anderson,Mark,mark.anderson46@example.com
15,2026-05-02T10:07:01.276Z,US,2023-07-15,USR_1014,partner,Miller,Daniel,daniel.miller14@example.com
20,2026-05-02T10:07:01.276Z,DE,2024-11-02,USR_1019,social_media,Lee,Susan,susan.lee19@example.com


## Remove Duplicate Users

This cell removes duplicate records based on the `user_id` column to ensure data uniqueness in the silver layer.

In [0]:
# Drop duplicates based on user_id column only
df_clean_nodup = df_clean.dropDuplicates(["user_id"])
display(df_clean_nodup)

_row,_fivetran_synced,country,signup_date,user_id,referral_source,last_name,first_name,email
22,2026-05-02T10:07:01.276Z,AU,2024-06-10,USR_1021,social_media,Rodriguez,Edward,edward.rodriguez21@example.com
27,2026-05-02T10:07:01.277Z,US,2022-02-07,USR_1026,partner,Ramirez,Emma,emma.ramirez26@example.com
32,2026-05-02T10:07:01.277Z,AU,2022-12-11,USR_1031,partner,Lopez,Barbara,barbara.lopez31@example.com
37,2026-05-02T10:07:01.277Z,AU,2023-07-11,USR_1036,partner,Garcia,Emma,emma.garcia36@example.com
5,2026-05-02T10:07:01.276Z,UK,2024-05-23,USR_1004,social_media,Miller,Emma,emma.miller4@example.com
42,2026-05-02T10:07:01.277Z,US,2022-06-02,USR_1041,referral,Davis,Patricia,patricia.davis41@example.com
10,2026-05-02T10:07:01.276Z,US,2024-02-29,USR_1009,partner,Jones,Edward,edward.jones9@example.com
47,2026-05-02T10:07:01.277Z,US,2022-05-05,USR_1046,organic,Anderson,Mark,mark.anderson46@example.com
15,2026-05-02T10:07:01.276Z,US,2023-07-15,USR_1014,partner,Miller,Daniel,daniel.miller14@example.com
20,2026-05-02T10:07:01.276Z,DE,2024-11-02,USR_1019,social_media,Lee,Susan,susan.lee19@example.com


## Save Cleaned Users to Silver Layer

This cell:
* Creates the `silver_cleaned_data` schema if it doesn't exist
* Writes the cleaned and deduplicated users data to the silver layer table `databrickscourse.silver_cleaned_data.users_cleaned`

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver_cleaned_data")
df_clean_nodup.write.format("delta").mode("overwrite").saveAsTable("databrickscourse.silver_cleaned_data.users_cleaned")

## Load Support Tickets from Bronze Layer

This cell loads the raw support tickets data from the bronze layer for inspection.

In [0]:
df_tickets = spark.table("databrickscourse.google_sheets.support_tickets")
display(df_tickets)

_row,_fivetran_synced,satisfaction_score,user_id,resolved_date,resolution_time_hours,created_date,ticket_id,category,priority
22,2026-05-02T10:16:37.246Z,5,USR_1031,2024-04-02,93,2024-03-30,TKT_030021,onboarding,high
27,2026-05-02T10:16:37.250Z,5,USR_1040,2024-04-23,576,2024-03-30,TKT_030026,account,high
32,2026-05-02T10:16:37.250Z,1,USR_1028,2024-04-19,70,2024-04-17,TKT_030031,billing,high
37,2026-05-02T10:16:37.251Z,5,USR_1024,2024-10-11,96,2024-10-07,TKT_030036,technical,low
5,2026-05-02T10:16:37.245Z,2,USR_1045,2024-05-04,336,2024-04-20,TKT_030004,onboarding,low
10,2026-05-02T10:16:37.245Z,null,USR_1004,null,null,2024-03-06,TKT_030009,technical,critical
15,2026-05-02T10:16:37.245Z,5,USR_1030,2024-06-09,430,2024-05-23,TKT_030014,billing,low
20,2026-05-02T10:16:37.246Z,5,USR_1048,2024-07-07,178,2024-06-30,TKT_030019,technical,critical
25,2026-05-02T10:16:37.246Z,4,USR_1037,2024-09-29,75,2024-09-26,TKT_030024,technical,low
30,2026-05-02T10:16:37.250Z,1,USR_1038,2024-12-14,509,2024-11-23,TKT_030029,feature_request,medium


## Save Support Tickets to Silver Layer

This cell writes the support tickets data to the silver layer table `databrickscourse.silver_cleaned_data.support_tickets_cleaned`.

In [0]:
df_tickets = spark.table("databrickscourse.google_sheets.support_tickets").write.format("delta").mode("overwrite").saveAsTable("databrickscourse.silver_cleaned_data.support_tickets_cleaned")